# 06 Erste (baseline) Modelle
|Index|Modell|
|---|---|
|L-lr-01|Logistische Regression (simple)|
|L-xbg-01|XGBoost (simple)|
|L-svm-01|Support Vector Machine (simple)|

## Import

In [1]:
import time

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier

import matplotlib as mpl
import matplotlib.pyplot as plt

In [2]:
mpl.style.use("seaborn-v0_8-colorblind")
RANDOM_STATE = 42

## Dataframes

In [3]:
df_raw = fetch_openml(data_id=42742, as_frame=True).frame

In [4]:
feat_cols = [col for col in df_raw.columns if col not in ("target",)]
cat_cols = [col for col in feat_cols if col.endswith("_cat")]
bin_cols = [col for col in feat_cols if col.endswith("_bin")]
num_cols = [col for col in feat_cols if col not in cat_cols and col not in bin_cols]

for col in cat_cols:
    df_raw[col] = df_raw[col].astype("category")

for col in bin_cols:
    if df_raw[col].isna().sum() == 0:
        df_raw[col] = df_raw[col].astype(int).astype("bool")
    else:
        df_raw[col] = df_raw[col].astype("Int8")

for col in num_cols:
    df_raw[col] = df_raw[col].astype("float32")

df_raw["target"] = df_raw["target"].astype(int).astype("bool")

idx_train = np.load("../../../data/processed/train_idx.npy")
idx_val = np.load("../../../data/processed/val_idx.npy")
idx_test = np.load("../../../data/processed/test_idx.npy")

df_train = df_raw.iloc[idx_train]
df_val = df_raw.iloc[idx_val]
df_test = df_raw.iloc[idx_test]

x_train, y_train = df_train[feat_cols], df_train["target"]
x_val, y_val = df_val[feat_cols], df_val["target"]
x_test, y_test = df_test[feat_cols], df_test["target"]

pd.Series(
    {
        "train": [len(df_train), len(x_train), len(y_train)],
        "val": [len(df_val), len(x_val), len(y_val)],
        "test": [len(df_test), len(x_test), len(y_test)]
    }
)

train    [476168, 476168, 476168]
val         [59522, 59522, 59522]
test        [59522, 59522, 59522]
dtype: object

## Hilfsvariablen

In [5]:
calc_cols = [c for c in feat_cols if c.startswith("ps_calc_")]

mv_cols = ["ps_car_03_cat", "ps_car_05_cat", "ps_reg_03", "ps_car_14"]

num_cols_no_calc = [c for c in num_cols if c not in calc_cols]
bin_cols_no_calc = [c for c in bin_cols if c not in calc_cols]

high_kard_cols = ["ps_car_11_cat"]
low_kard_cols = [c for c in cat_cols if c not in high_kard_cols]

## L-lr-01

In [6]:
# Preprocessing mit calc 

preproc_logregMK2 = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler())
    ]), num_cols),
    ("cat_onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False), low_kard_cols),
    ("cat_target", TargetEncoder(cv=5, shuffle=True, random_state=RANDOM_STATE), high_kard_cols),
    ("bin", "passthrough", bin_cols)
], remainder="drop").set_output(transform="pandas")

In [7]:
# Näher an VL

logreg_01 = make_pipeline(
    make_column_transformer(
        (
            make_pipeline(
                SimpleImputer(strategy="median", add_indicator=True),
                StandardScaler()
            ),
            num_cols
        ),
        (
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            low_kard_cols
        ),
        (
            TargetEncoder(cv=5, shuffle=True, random_state=RANDOM_STATE),
            high_kard_cols
        ),
        (
            "passthrough",
            bin_cols
        )
    ),
    LogisticRegression(
        max_iter=1000, 
        random_state=RANDOM_STATE
    )
)

## L-xgb-01

In [8]:
# Nativ Klassen und nan verarbeiten

xgb_01 = XGBClassifier(
    booster="gbtree",
    random_state=RANDOM_STATE,
    eval_metric="auc",
    tree_method="hist",
    enable_categorical=True
)

In [9]:
# mit encodings

xgb_01_enc = make_pipeline(
    make_column_transformer(
        ("passthrough", num_cols),
        (OneHotEncoder(handle_unknown="ignore", sparse_output=False), low_kard_cols),
        (TargetEncoder(cv=5, shuffle=True, random_state=RANDOM_STATE), high_kard_cols),
        ("passthrough", bin_cols)
    ),
    XGBClassifier(
        booster="gbtree",
        random_state=RANDOM_STATE,
        eval_metric="auc",
        tree_method="hist"
    )
)

## L-svm-01

In [10]:
svm_01 = make_pipeline(
    make_column_transformer(
        (
            make_pipeline(
                SimpleImputer(strategy="median", add_indicator=True),
                StandardScaler()
            ),
            num_cols
        ),
        (
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            low_kard_cols
        ),
        (
            TargetEncoder(cv=5, shuffle=True, random_state=RANDOM_STATE),
            high_kard_cols
        ),
        (
            "passthrough",
            bin_cols
        )
    ),
    LinearSVC(
        max_iter=1000, 
        random_state=RANDOM_STATE
    )
)

## Training

In [11]:
models = {
    "logreg_01": logreg_01,
    "xgb_01": xgb_01,
    "xgb_01_enc": xgb_01_enc,
    "svm_01": svm_01
}

In [12]:
train_times = {}

for name, model in models.items():
    start_time = time.time()
    model.fit(x_train, y_train)
    train_times[name] = time.time() - start_time
    print(f"Training time for {name}: {train_times[name]:.2f} seconds")

c:\Users\Linus Lauschke\anaconda3\envs\ADA\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


Training time for logreg_01: 5.35 seconds
Training time for xgb_01: 9.86 seconds


c:\Users\Linus Lauschke\anaconda3\envs\ADA\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


Training time for xgb_01_enc: 8.35 seconds


c:\Users\Linus Lauschke\anaconda3\envs\ADA\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


Training time for svm_01: 44.69 seconds


## Evaluation

In [15]:
# helper

def eval_model(name, model, x, y):
    scores = model.predict_proba(x)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(x)
    auc = roc_auc_score(y, scores)
    return {"modell": name, "auc": auc, "gini": 2 * auc - 1, "trainingszeit_s": train_times[name]}

results = []

In [16]:
# logreg01
res_logreg = eval_model("logreg_01", logreg_01, x_val, y_val)
results.append(res_logreg)
res_logreg

{'modell': 'logreg_01',
 'auc': 0.6168062891547132,
 'gini': 0.2336125783094265,
 'trainingszeit_s': 5.351555109024048}

In [17]:
# xgb01
res_xgb = eval_model("xgb_01", xgb_01, x_val, y_val)
results.append(res_xgb)
res_xgb

{'modell': 'xgb_01',
 'auc': 0.598312239013963,
 'gini': 0.19662447802792604,
 'trainingszeit_s': 9.86265516281128}

In [18]:
#xgb01_enc
res_xgb_enc = eval_model("xgb_01_enc", xgb_01_enc, x_val, y_val)
results.append(res_xgb_enc)
res_xgb_enc

{'modell': 'xgb_01_enc',
 'auc': 0.6173498440582039,
 'gini': 0.23469968811640785,
 'trainingszeit_s': 8.35004186630249}

In [19]:
# svm01
res_svm = eval_model("svm_01", svm_01, x_val, y_val)
results.append(res_svm)
res_svm

{'modell': 'svm_01',
 'auc': 0.617425106124739,
 'gini': 0.234850212249478,
 'trainingszeit_s': 44.69226026535034}

In [20]:
df_results = pd.DataFrame(results).set_index("modell")
df_results

,auc,gini,trainingszeit_s
modell,,,
logreg_01,0.616806,0.233613,5.351555
xgb_01,0.598312,0.196624,9.862655
xgb_01_enc,0.617350,0.234700,8.350042
svm_01,0.617425,0.234850,44.692260


## Notizen

- Gini ist nur eine lineare transformation der AUC. Also können wir eigentlich beides wählen. Wenn wir uns möglichst an die Kaggle Competition halten wollen, benutzen wir Gini. Dennoch unterscheidet es sich leicht von der normalized Gini von Kaggle, da hier durch einen bestmöglichen Gini geteilt wird. 
- Wenn die Zeit bleibt schaue ich mal, ob AUC oder Gini einen unterschiedlichen einfluss auf Hyperparameter haben.


- Zunächst nur Baseline also ertsmal einfach ganz simples Preprocessing, Modell training pipeline. To beexpanded upon. Lernkurven etc im späteren verlauf.


- Ich nehme 01 als komplette baseline also keinerlei modellanpassungen. Classweights, Sampling etc später.